<center><b>Python Programming for Multilingual Text</b></center>
<center>3-2: Historical Named Entity Recognition with Stanza</center>

---

# Named Entity Recognition with Stanza

## Overview

In 3-1 we scaled our text analysis tools up from a single letter to the whole Montagu corpus, and along the way we ran into POS tagging — assigning grammatical categories to words. This notebook introduces a related but different task: **Named Entity Recognition (NER)**.

Where POS tagging asks "what part of speech is this word?", NER asks "does this word refer to a person, a place, an organization, a date, or something else in the world?"

We will follow the same pattern as 3-1: start with one letter, understand the tool deeply, then scale up to the full corpus. New to this notebook:

1. **Setup**: installing and loading Stanza
2. **NER on a single sentence**: understanding entity types and the output format
3. **NER on Letter 35**: applying it to a full letter
4. **A key limitation**: why a model trained on modern news text struggles with 18th-century letters
5. **Scaling to the corpus**: running NER on all letters and storing results
6. **Aggregating entities**: counting people and places across the corpus

---

**Dataset:** Lady Mary Wortley Montagu's *Turkish Embassy Letters* (1716–1718)

---
## Step 1: Setup

[Stanza](https://stanfordnlp.github.io/stanza/ner.html) is a different library from NLTK, so it has its own installation and download steps. Unlike NLTK, where we downloaded small individual resources (`punkt`, `stopwords`, etc.), Stanza downloads a full **pipeline** which is a bundle of neural network models that work together (tokenizer, POS tagger, NER tagger, and so on).

In [ ]:
import stanza
import pandas as pd
from collections import Counter
from itertools import chain

stanza.download('en') # download English model

In [ ]:
# Let's test the example from the repository itself
# https://github.com/stanfordnlp/stanza
# This allows us to test that the code is working as intended.

nlp = stanza.Pipeline('en') # This sets up a default neural pipeline in English
doc = nlp("Barack Obama was born in Hawaii. He was elected president in 2008.")
doc.sentences[0].print_dependencies()

> **What just happened?** `stanza.Pipeline(...)` loaded the models into memory and built a function-like object, `nlp`, that we can now call on any piece of text. This loading step is the slow part — once `nlp` exists, processing text with it is much faster. This is why we build the pipeline once, outside of any loop, rather than rebuilding it for every letter.

In [ ]:
# processors='tokenize,ner' loads only what we need:
# - tokenize: splits text into words and sentences (NER depends on this)
# - ner: the named entity recognizer itself

# We would run this if we had not run the cell 
# the main difference is that this code would only load the tokenizer, which we need to be able to run the NER and the NER model

nlp = stanza.Pipeline(lang='en', processors='tokenize,ner')

---
## Step 2: NER on a single sentence

Before we turn to a the letters, let's pick a sentence from one letter and run NER on it

In [ ]:
# it is a sentence from letter 32
sample = "My side-saddle is the first that was ever seen in this part of the world, and is gazed at with as much wonder as the ship of Columbus in the first discovery of America."

doc = nlp(sample)

for sentence in doc.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}") #!r:25 makes print statement look neat like a table

Let's unpack this:

`doc = nlp(sample)` -> running our pipeline on a string returns a `Document` object, not a plain string or list. This is different from NLTK, where most functions returned lists or tuples directly.

`doc.sentences` -> a `Document` is broken into `Sentence` objects, even if your input is one sentence. This matters for the corpus-level work later, since Stanza's own sentence splitting can disagree with NLTK's `sent_tokenize()`.

`sentence.ents` -> each `Sentence` has an `.ents` attribute: a list of entities Stanza found in it.

`ent.text` and `ent.type` -> each entity has the original text span (`ent.text`) and a category label (`ent.type`), such as `PERSON`, `GPE` (geo-political entity — countries, cities, states), `ORG`, or `DATE`.

In [ ]:
# this print statement from stanza documentation shows us each word in the sentence and how it was/was not tagged

print(*[f'token: {token.text}\tner: {token.ner}' for sent in doc.sentences for token in sent.tokens], sep='\n')

### Entity types you'll commonly see

| Type | Meaning | Example |
|---|---|---|
| `PERSON` | People, including fictional | *Obama, Montagu* |
| `GPE` | Countries, cities, states | *Hawaii, Constantinople* |
| `LOC` | Non-GPE locations: mountains, bodies of water, regions | *the Bosphorus* |
| `ORG` | Companies, agencies, institutions | *the Senate* |
| `DATE` | Absolute or relative dates | *1717, last Tuesday* |
| `NORP` | Nationalities, religious or political groups | *French, Catholic* |
| `FAC` | Buildings, airports, bridges | *the Globe Theatre* |


> Stanza's English NER model is trained on the **[OntoNotes](https://catalog.ldc.upenn.edu/LDC2013T19)** dataset, a large collection of modern news, talk shows, web text, and similar sources. They have 18 categories for entities, which you can read more about [here on page 21](https://catalog.ldc.upenn.edu/docs/LDC2013T19/OntoNotes-Release-5.0.pdf)

---
## Step 3: NER on Letter 35

Let's bring back Letter 35, the same letter we worked with closely in 2-4 and again at the start of 3-1.

In [ ]:
# Load the corpus, same as in 3-1
montagu = pd.read_csv('../../output/montagu_letters_v3.csv')
montagu = montagu[['filename', 'title', 'body']]

letter_35 = montagu['body'].iloc[34]
print(letter_35[:300])

In [ ]:
# Run the Stanza pipeline on the full letter
doc_35 = nlp(letter_35)

print(f"Number of sentences Stanza found: {len(doc_35.sentences)}")

> Compare this sentence count to the one we got from NLTK's `sent_tokenize()` in 3-1 (68, for Letter 35). Are they the same? If not, why might two different tools disagree about where sentences begin and end?

In [ ]:
### YOUR CODE HERE
# Extract all entities from doc_35 and print them as (text, type) pairs,
# the same way we did for the sample sentence above.


In [ ]:
# This is how I would do it

letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]

for text, etype in letter_35_entities:
    print(f"{text!r:30} {etype}")

Let's unpack the list comprehension:

```python
letter_35_entities = [(ent.text, ent.type) for sentence in doc_35.sentences for ent in sentence.ents]
```

This is a **nested** list comprehension — two `for` clauses instead of one. Read it left to right, but think about it inside-out:

`for sentence in doc_35.sentences` -> outer loop, over each sentence in the document

`for ent in sentence.ents` -> inner loop, over each entity within that sentence

`(ent.text, ent.type)` -> what we keep from each entity: the text span and its label

This is equivalent to:

```python
letter_35_entities = []
for sentence in doc_35.sentences:
    for ent in sentence.ents:
        letter_35_entities.append((ent.text, ent.type))
```

Both versions do the same thing. The nested comprehension is more compact once you're comfortable with it, but there's no shame in writing the longer version first and condensing it later.

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Look closely at the output above. Are there any entities that are clearly wrong — mislabeled, or not entities at all? Are there any real people or places in the letter that Stanza missed entirely?
</div>

---
## Step 4: A key limitation — why this matters for historical text

Stanza's English NER model was trained on OntoNotes: modern news wire, broadcast conversation, web text, and similar genres from the late 20th and early 21st century. Montagu was writing private letters in 1716–1718, describing a part of the world with person and place names with several competing English spellings that does not appear in the training data.

Researchers who work on **historical NER** as a subfield have documented this gap extensively. The challenges they describe consistently include archaic spelling and inconsistent orthography, named entities that no longer exist or have changed meaning, and a basic shortage of historical text to train on compared to the enormous amount of modern text available online. None of this is a flaw in Stanza specifically — it's a structural feature of how all modern NER models get trained, and it generalizes to spaCy, FLAIR, or any other off-the-shelf system you might reach for.

> This is not a reason to abandon NER for historical text. It's a reason to treat its output as a **starting point for inquiry**, not a finished answer. We need to read the output critically to ensure that we are actually we are actually capturing what is valuable and interesting to us.

In [ ]:
# Let's look at a sentence built specifically to surface this gap.
# Try swapping in a sentence or two of your own choosing from Letter 35.

test_sentence = "We lay that night at a town called Bujuk Cekmege, or Great Bridge; and the night following, at Kujuk Cekmege, or Little Bridge; in a very pleasant lodging, formerly a monastery of dervises; having before it a large court, encompassed with marble cloisters, with a good fountain in the middle."

doc_test = nlp(test_sentence)
for sentence in doc_test.sentences:
    for ent in sentence.ents:
        print(f"{ent.text!r:25} {ent.type}")

---
## Step 5: Scaling to the corpus

Now that we have looked closely at one letter, let's apply the same pipeline to all of Montagu's letters — the same move 3-1 made for word counts, POS tags, and lemmas.

> ⚠️ Running a neural pipeline on every letter in the corpus is much slower than the simple string operations from 3-1. Expect this cell to take noticeably longer to run. This is normal — Stanza is doing far more computation per word than `.split()` or even `word_tokenize()`.

In [ ]:
# Run NER on every letter and store the resulting Document object
montagu['ner_doc'] = montagu['body'].apply(lambda text: nlp(text))

It took ca. 19 minutes to run

We've stored the full `Document` object for each letter, not just the entities. This mirrors what we did with `pos_tags` in 3-1 — keep the richer, intermediate result around so you don't have to recompute it if you want to extract something different from it later.

Now let's pull just the entities out into their own column, the same way we pulled `nouns` out of `pos_tags` in 3-1.

In [ ]:
def extract_entities(doc):
    """Extract all (text, type) entity pairs from a Stanza Document."""
    return [(ent.text, ent.type) for sentence in doc.sentences for ent in sentence.ents]

montagu['entities'] = montagu['ner_doc'].apply(extract_entities)
montagu[['title', 'entities']].head()

In [ ]:
montagu.info()

In [ ]:
# Save the enriched DataFrame. Note: the 'ner_doc' column contains Stanza Document
# objects, not plain text, so we drop it before saving to CSV — it won't serialize cleanly.
montagu_to_save = montagu.drop(columns=['ner_doc'])
montagu_to_save.to_csv('../../output/montagu_letters_v4.csv', index=False)
print("Saved! Columns now:", montagu_to_save.columns.tolist())

---
## Step 6: Aggregating entities across the corpus

With entities extracted for every letter, we can ask corpus-level questions: who does Montagu mention most? What places recur across her letters?

### People

In [ ]:
# Flatten all entities into one list, same pattern as chain.from_iterable in 3-1
all_entities = list(chain.from_iterable(montagu['entities']))

# Keep only PERSON entities
people = [text for text, etype in all_entities if etype == 'PERSON']
person_counts = Counter(people)

print("Top 20 most-mentioned people:")
for name, count in person_counts.most_common(20):
    print(f"  {name}: {count}")

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  Scan this list. Are these all genuinely distinct people, or are they even all people?
</div>

### Places

In [ ]:
### YOUR CODE HERE
# Following the pattern above, extract GPE and LOC entities and count the most common ones.


In [ ]:
# This is how I would do it

places = [text for text, etype in all_entities if etype in ('GPE', 'LOC')]
place_counts = Counter(places)

print("Top 20 most-mentioned places:")
for place, count in place_counts.most_common(20):
    print(f"  {place}: {count}")

### A quick sanity check

Before trusting these counts for any real argument about the corpus, it's worth checking a handful of entries by hand against the original letters. Pick two or three names or places from the top of each list above and search for them directly in the text.

In [ ]:
# Pick one place from your list above and confirm it actually appears where you'd expect
search_term = "Vienna"  # change this to check on a new term

matches = montagu[montagu['body'].str.contains(search_term, case=False, na=False)]
print(f"'{search_term}' appears in {len(matches)} letter(s):")
print(matches['title'].tolist())